# Librerías

In [2]:
import numpy as np
from datasets import load_dataset
!pip install evaluate
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00


# Dataset usado

In [3]:
# Cargamos el dataset https://huggingface.co/datasets/mteb/SpanishSentimentClassification
ds = load_dataset("mteb/SpanishSentimentClassification")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/59.6k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1029 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/147 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/296 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1029
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 147
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 296
    })
})

In [4]:
# Ejemplos
display(ds["train"].shuffle(100).select(range(10)).to_pandas())
display(ds["test"].shuffle(100).select(range(10)).to_pandas())

,text,label
0,El SPA es genial .,1
1,Mala estancia,0
2,Viaje placentero,1
3,Dias placenteros,1
4,Todo MUY limpio y cuidado .,1
5,A primera hora de la mañana la habitación pued...,0
6,Lo barato sale caro,0
7,El trabajo de Kenia al frente de los espectácu...,1
8,Las instalaciones bastante bien : habitación a...,1
9,Había una caja fuerte en el armario sin cargo .,1


,text,label
0,Allí todo el personal me trató muy bien y pude...,1
1,Amable bienvenida en la recepción .,1
2,Habitación sencilla pero muy lipia con baño su...,1
3,Un hotel de ciudad acogedor y muy bien situado .,1
4,Gestión nefasta,0
5,La estancia en el hotel la verdad es que fue a...,1
6,hotel situado al lado del camp nou con transpo...,1
7,"No había ni un salon estar para los clientes ,...",0
8,Muy adecuado para descansar .,1
9,La verdad que merece la pena reservar media pe...,1


In [5]:
from collections import Counter

# Conteo en train
train_counts = Counter(ds["train"]["label"])
test_counts  = Counter(ds["test"]["label"])

print("TRAIN:")
print("  #0:", train_counts[0])
print("  #1:", train_counts[1])

print("\nTEST:")
print("  #0:", test_counts[0])
print("  #1:", test_counts[1])

TRAIN:
  #0: 178
  #1: 851

TEST:
  #0: 52
  #1: 244


# Preprocesado

In [6]:
# Descargamos el tokenizador
model_name = "BSC-LT/mRoBERTa"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Longitud máxima de secuencia
max_length = 256

# El tokenizador devolverá el texto tokenizado y mapeado a IDs de vocabulario
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

# tokenizamos todo el dataset
encoded = ds.map(preprocess, batched=True)
# renombramos la columna 'label' a 'labels' para que el Trainer lo reconozca
encoded = encoded.rename_column("label", "labels")
# nos quedamos solo con las columnas que nos interesan
encoded = encoded.remove_columns([c for c in encoded["train"].column_names
                                 if c not in ["input_ids", "attention_mask", "labels"]])

# inputs_ids: ids de los tokens
# attention_mask: máscara de atención
# labels: etiquetas de clase
encoded


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Map:   0%|          | 0/1029 [00:00<?, ? examples/s]

Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/296 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1029
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 147
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 296
    })
})

In [7]:
print(encoded["train"]["input_ids"][0])
print(tokenizer.decode(encoded["train"]["input_ids"][0]))
print(encoded["train"]["attention_mask"][0])
print(f"Label: {encoded['train']['labels'][0]}")

[0, 45490, 78204, 499, 573, 11211, 428, 515, 19587, 1413, 569, 5177, 1239, 3617, 109777, 398, 1078, 95889, 472, 1407, 70917, 1413, 875, 33130, 3969, 1413, 510, 73358, 1038, 31773, 12925, 726]
<s> Está situado en el centro de la ciudad , con todo lo más turístico a tu alrededor ( El Pilar , por ejemplo ) , y sitios para tomar algo .
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Label: 1


# El modelo

In [8]:
num_labels = 2
id2label = {0: "NEG", 1: "POS"}
label2id = {"NEG": 0, "POS": 1}

# Descargamos el modelo preentrenado y le ponemos una capa de clasificación de dos clases
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: BSC-LT/mRoBERTa
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Entrenamiento

In [9]:
# Métricas de evaluación: accuracy, macro-F1, precision/recall/F1 para la clase POS=0
acc = evaluate.load("accuracy")
f1_macro = evaluate.load("f1")
prec = evaluate.load("precision")
rec = evaluate.load("recall")

# función que se le pasará al Trainer para computar las métricas
# recibe un par (logits, labels)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # macro-F1 (robusto si hay desbalanceo)
    out = {}
    out.update(acc.compute(predictions=preds, references=labels))
    out["f1_macro"] = f1_macro.compute(predictions=preds, references=labels, average="macro")["f1"]

    # binary precision/recall/f1 tomando POS=1 como clase positiva
    out["precision_pos"] = prec.compute(predictions=preds, references=labels, average="binary", pos_label=0)["precision"]
    out["recall_pos"] = rec.compute(predictions=preds, references=labels, average="binary", pos_label=0)["recall"]
    out["f1_pos"] = f1_macro.compute(predictions=preds, references=labels, average="binary", pos_label=0)["f1"]

    return out

In [10]:
from transformers import EarlyStoppingCallback

# Data collator para hacer padding dinámico
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Parámetros de entrenamiento
args = TrainingArguments(
    output_dir="mroberta-spanish-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", # métrica para seleccionar el mejor modelo, dataset desbalanceado
    greater_is_better=True,
    logging_strategy="epoch",
    logging_steps=1,
    report_to="none",
    save_total_limit=2,
)

# Creamos el Trainer
# Se encarga de entrenar el modelo y evaluar en cada epoch
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # <-- early stopping
)


In [11]:
# entrenamos
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Pos,Recall Pos,F1 Pos
1,0.437424,0.289025,0.918367,0.839636,0.888889,0.615385,0.727273
2,0.216645,0.202951,0.952381,0.908166,1.000000,0.730769,0.844444
3,0.134587,0.230082,0.938776,0.893246,0.840000,0.807692,0.823529
4,0.094583,0.117594,0.972789,0.951803,0.958333,0.884615,0.920000
5,0.077591,0.133030,0.965986,0.938776,0.956522,0.846154,0.897959
6,0.047401,0.141611,0.972789,0.950203,1.000000,0.846154,0.916667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=390, training_loss=0.16803849904965132, metrics={'train_runtime': 486.7635, 'train_samples_per_second': 21.14, 'train_steps_per_second': 1.335, 'total_flos': 192966778556340.0, 'train_loss': 0.16803849904965132, 'epoch': 6.0})

In [12]:
from pprint import pprint
# Podemos evaluar el modelo en el conjunto de test
test_metrics = trainer.evaluate(encoded["test"])
pprint(test_metrics)


{'epoch': 6.0,
 'eval_accuracy': 0.9594594594594594,
 'eval_f1_macro': 0.9254032258064516,
 'eval_f1_pos': 0.875,
 'eval_loss': 0.17544570565223694,
 'eval_precision_pos': 0.9545454545454546,
 'eval_recall_pos': 0.8076923076923077,
 'eval_runtime': 1.7729,
 'eval_samples_per_second': 166.959,
 'eval_steps_per_second': 5.641}


In [13]:
# Vamos a usar el modelo para hacer predicciones
from transformers import pipeline

# para predicción end-to-end se suele usar pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=0)

raw_test = ds["test"].shuffle(100)
for i in range(8):
    text = raw_test[i]["text"]
    gold = raw_test[i]["label"]
    pred = clf(text)[0]
    print(f"\n[{i + 1}] gold={gold} text: {text[:180]}")
    pprint(pred)



[1] gold=1 text: Allí todo el personal me trató muy bien y pude disfrutar de las muchas comodidades del recinto .
{'label': 'POS', 'score': 0.988200306892395}

[2] gold=1 text: Amable bienvenida en la recepción .
{'label': 'POS', 'score': 0.9880344271659851}

[3] gold=1 text: Habitación sencilla pero muy lipia con baño suficiente con amenities
{'label': 'POS', 'score': 0.9880863428115845}

[4] gold=1 text: Un hotel de ciudad acogedor y muy bien situado .
{'label': 'POS', 'score': 0.9881652593612671}

[5] gold=0 text: Gestión nefasta
{'label': 'NEG', 'score': 0.9621155858039856}

[6] gold=1 text: La estancia en el hotel la verdad es que fue agradable y cumplió con lo que ibamos buscando .
{'label': 'POS', 'score': 0.9881530404090881}

[7] gold=1 text: hotel situado al lado del camp nou con transporte publico enfrente del hotel que te deja en todos los lugares turisticos de barcelona , el hote como todos los senator ESTUPENDO hab
{'label': 'POS', 'score': 0.9882426857948303}

[8] gold=0